# Hansen Ch.7 习题解答（计算部分）

**Chapter 7 Asymptotic Theory for Least Squares**

完整推导与**面向初学者的详细注释**见同目录 `Hansen_Ch07_Exercises_Solutions.md`（强烈建议先读 §0、§1）。

本 notebook：**Exercise 7.4**（矩核对）+ **Exercise 7.28**（delta method 实证）。

> **写给只学过李子奈/陈强的同学：** 本章是全书枢纽——**去掉第 5 章的正态假定**，只用 WLLN + CLT 在大样本下重建 OLS 的性质。两条定律：**WLLN** 给**一致性**（$\hat\beta\to_p\beta$），**CLT** 给**渐近正态**。
> 全章的核心是一个分解：
> $$\sqrt n(\hat\beta-\beta)=\underbrace{\hat Q^{-1}}_{\to_p Q^{-1}}\cdot\underbrace{\tfrac1{\sqrt n}\textstyle\sum X_ie_i}_{\xrightarrow{\text{CLT}}N(0,\Omega)}\to_d N(0,Q^{-1}\Omega Q^{-1}).$$
> 这正是第 4 章的**夹心方差**，现在以**极限**形式出现（同方差下退化为 $\sigma^2Q^{-1}$）。
> **delta method** 估计非线性参数 $\theta=g(\beta)$：$\sqrt n(\hat\theta-\theta)\to_d N(0,G'V_\beta G)$，$G=\nabla g$。下面 7.28 的"教育/经验回报比"、回归函数 CI、预测区间都用它。


## Exercise 7.4 矩核对（数值）

In [ ]:
import numpy as np
# X1,X2 in {-1,1} with given joint probs
# outcomes: (1,1),(1,-1),(-1,1),(-1,-1) with probs 3/8,1/8,1/8,3/8
vals = np.array([[1,1],[1,-1],[-1,1],[-1,-1]], float)
p = np.array([3/8,1/8,1/8,3/8])
X1, X2 = vals[:,0], vals[:,1]
same = (X1==X2)
sig2 = np.where(same, 5/4, 1/4)

print('E[X1]=', np.sum(p*X1))
print('E[X1^2]=', np.sum(p*X1**2))
print('E[X1X2]=', np.sum(p*X1*X2))
print('E[e^2]=', np.sum(p*sig2))
print('E[X1^2 e^2]=', np.sum(p*(X1**2)*sig2))
print('E[X1X2 e^2]=', np.sum(p*X1*X2*sig2))
print('targets: 0,1,0.5,1,1,0.875')


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

CPS = Path("../../hansen/econometrics/data/cps09mar/cps09mar.xlsx")
if not CPS.exists():
    CPS = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data/cps09mar/cps09mar.xlsx")
df = pd.read_excel(CPS)
df["experience"] = df["age"] - df["education"] - 6
df["lwage"] = np.log(df["earnings"]/(df["hours"]*df["week"]))
df["exp2"] = (df["experience"]**2)/100
s = df[(df.race==1)&(df.female==0)&(df.hisp==1)].copy()
y = s.lwage.to_numpy(float)
X = np.c_[s.education, s.experience, s.exp2, np.ones(len(s))]
names = ["education","experience","exp2/100","intercept"]
n,k = X.shape
beta = np.linalg.lstsq(X,y,rcond=None)[0]
e = y - X@beta
XXinv = np.linalg.inv(X.T@X)
h = np.sum(X*(X@XXinv), axis=1)
u = X * (e/(np.clip(1-h,1e-12,None)))[:,None]  # HC3
V = XXinv @ (u.T@u) @ XXinv
se = np.sqrt(np.diag(V))
print(pd.DataFrame({"beta":beta,"HC3_SE":se}, index=names))
print("n=", n)


### (b)–(d) $\theta=\beta_1/(\beta_2+0.2\beta_3)$（experience=10 时教育/经验回报比）

**delta method：** $\theta=g(\beta)$，渐近 $\sqrt n(\hat\theta-\theta)\to_d N(0,G'V_\beta G)$，$G=\nabla g$。
本题 $g=\beta_1/d$，$d=\beta_2+0.2\beta_3$（=经验在 exp=10 处的边际效应 $\beta_2+2\beta_3\cdot 10/100$）。
梯度 $G=(1/d,\;-\beta_1/d^2,\;-0.2\beta_1/d^2,\;0)$，标准误 $\widehat{\mathrm{se}}=\sqrt{\hat G'\hat V\hat G}$。


In [ ]:
b1,b2,b3,b0 = beta
den = b2 + 0.2*b3
theta = b1/den
grad = np.array([1/den, -b1/den**2, -0.2*b1/den**2, 0.0])
se_theta = float(np.sqrt(grad @ V @ grad))
z90 = stats.norm.ppf(0.95)
print(f"theta = {theta:.4f}")
print(f"se(theta) = {se_theta:.4f}")
print(f"90% CI = [{theta - z90*se_theta:.4f}, {theta + z90*se_theta:.4f}]")


### (e) 回归函数在 education=12, experience=20

In [ ]:
x = np.array([12.0, 20.0, 4.0, 1.0])  # exp2/100 = 400/100=4
m = float(x @ beta)
se_m = float(np.sqrt(x @ V @ x))
z95 = 1.96
print(f"m(12,20) = {m:.4f}")
print(f"95% CI = [{m-z95*se_m:.4f}, {m+z95*se_m:.4f}]")


### (f) 样本外预测：edu=16, exp=5，80% 预测区间

In [ ]:
s2 = float(np.sum(e**2)/(n-k))
x = np.array([16.0, 5.0, 0.25, 1.0])
yhat = float(x @ beta)
se_f = float(np.sqrt(x @ V @ x + s2))
z80 = stats.norm.ppf(0.90)
lo, hi = yhat - z80*se_f, yhat + z80*se_f
print(f"point forecast log wage = {yhat:.4f}")
print(f"80% PI log wage = [{lo:.4f}, {hi:.4f}]")
print(f"80% PI wage = [{np.exp(lo):.2f}, {np.exp(hi):.2f}]")


## 理论结论的蒙特卡洛验证（无需外部数据）

以下单元格核对 ch07 的一致性/plim（7.1 短回归 OVB、7.24 衰减偏差）、非标准估计量方差（7.15）、delta method CI 覆盖。可独立运行。

In [ ]:
import numpy as np
rng = np.random.default_rng(7)

# Ex 7.1: 短回归 plim ≠ β（大样本 OVB）
n = 200000
X1 = rng.standard_normal(n)
X2 = 0.5 * X1 + rng.standard_normal(n)
Y = 1.0 * X1 + 2.0 * X2 + rng.standard_normal(n)
b1s = np.sum(X1 * Y) / np.sum(X1**2)
plim = 1.0 + np.mean(X1 * X2) / np.mean(X1**2) * 2.0
print(f"[7.1] 短回归 β̂₁={b1s:.3f}, 公式 plim={plim:.3f}  (≠ 真 β₁=1, OVB)")

# Ex 7.24: 自变量乘性测量误差 → 衰减偏差
Xstar = np.abs(rng.standard_normal(n)) + 0.5
v = rng.uniform(0.5, 1.5, n)
Y = 2.0 * Xstar + rng.standard_normal(n)
X = Xstar * v
bhat = np.sum(X * Y) / np.sum(X**2)
plim = 2.0 * np.mean(v) / np.mean(v**2)
print(f"[7.24] β̂={bhat:.3f}, plim=β·E[v]/E[v²]={plim:.3f}  (< β=2, 衰减)")

# Ex 7.15: 非标准估计量 ΣX³Y/ΣX⁴ 的渐近方差 = E[X⁶e²]/(E[X⁴])²
N, reps = 80, 4000
mhats = []
Xall = rng.standard_normal((reps, N)); eall = rng.standard_normal((reps, N))
for r in range(reps):
    Xs = Xall[r]; Ys = Xs + eall[r]
    mhats.append(np.sum(Xs**3 * Ys) / np.sum(Xs**4))
X = rng.standard_normal(400000); e = rng.standard_normal(400000)
avar = np.mean(X**6 * e**2) / (np.mean(X**4))**2
print(f"[7.15] var(β̂)·N={np.var(mhats)*N:.3f} ≈ 公式 avar={avar:.3f}")

# delta method: θ=β₁β₂ 的 95% CI 覆盖应≈0.95
n, reps = 200, 20000
b1, b2, theta = 1.0, 0.5, 0.5
cov = 0
for r in range(reps):
    X = np.c_[rng.standard_normal(n), rng.standard_normal(n), np.ones(n)]
    e = rng.standard_normal(n); Y = X @ [b1, b2, 0.0] + e
    bh = np.linalg.solve(X.T @ X, X.T @ Y)
    V = np.linalg.inv(X.T @ X)
    G = np.array([b2, b1, 0.0])
    se = np.sqrt(G @ V @ G)
    lo, hi = bh[0]*bh[1] - 1.96*se, bh[0]*bh[1] + 1.96*se
    cov += (lo <= theta <= hi)
print(f"[delta] θ=β₁β₂ 的 95% CI 覆盖={cov/reps:.4f}  (应≈0.95)")